# News Sentiment — Prototype Notebook

A scratch space for trying out the project's building blocks directly, one call at a time: fetching prices and earnings dates from Yahoo Finance, pulling news headlines for a ticker, and scoring those headlines with FinBERT and Qwen side by side.

This notebook isn't a feature — it's for poking at the APIs and sanity-checking behavior before anything new gets wired into `dashboard.py` or `cli.py`. Change `TICKER` below and re-run to explore a different stock.

Run it with `uv run jupyter lab` from the project root, or open it in VS Code / PyCharm with the project's `.venv` selected as the kernel.

In [ ]:
from datetime import date

import pandas as pd

from news_sentiment.earnings import YFinanceEarningsSource, is_earnings_soon
from news_sentiment.news_source import YFinanceNewsSource
from news_sentiment.prices import YFinancePriceSource
from news_sentiment.rating import aggregate_rating
from news_sentiment.sentiment import FinbertSentimentScorer, QwenSentimentScorer

TICKER = "NVDA"
HEADLINE_COUNT = 10

## 1. Price history and earnings date (yfinance)

Two layers here: the raw `yfinance` call, so you can see exactly what Yahoo Finance returns, and the project's own wrapper (`YFinancePriceSource`, `YFinanceEarningsSource`) that turns it into clean, typed values.

In [ ]:
import yfinance as yf

raw_ticker = yf.Ticker(TICKER)

raw_history = raw_ticker.history(period="5d")[["Close"]]
print("Raw price history (last 5 trading days):")
display(raw_history)

raw_calendar = raw_ticker.calendar
print("\nRaw earnings calendar dict:")
print(raw_calendar)

In [ ]:
price_source = YFinancePriceSource()
returns = price_source.fetch(TICKER)
print(returns)

earnings_source = YFinanceEarningsSource()
next_earnings = earnings_source.fetch(TICKER)
print(f"\nNext earnings date: {next_earnings}")
print(f"Earnings within 14 days of today? {is_earnings_soon(next_earnings, date.today(), days=14)}")

## 2. News headlines for the ticker

Same two-layer pattern: the raw article dict `yfinance` hands back, then `YFinanceNewsSource.fetch()`, which parses it into a clean list of `Headline` objects.

In [ ]:
raw_articles = raw_ticker.get_news(count=3)
print("One raw article dict:")
raw_articles[0] if raw_articles else "No articles returned"

In [ ]:
news_source = YFinanceNewsSource()
headlines = news_source.fetch(TICKER, count=HEADLINE_COUNT)

headlines_df = pd.DataFrame(
    [{"title": h.title, "publisher": h.publisher, "published_at": h.published_at} for h in headlines]
)
headlines_df

## 3. Sentiment scoring — FinBERT vs. Qwen

Both scorers implement the same interface (`score_all(headlines) -> list[ScoredHeadline]`) but work completely differently under the hood:

- **FinBERT** is a classifier fine-tuned on financial text — one forward pass per headline returns a label directly, with a calibrated confidence.
- **Qwen** is a general-purpose instruction-following LLM — it's prompted to answer Positive/Negative/Neutral in free text, which then has to be parsed back into a label.

Loading Qwen's weights the first time takes a minute; after that they're cached locally (`~/.cache/huggingface`).

In [ ]:
finbert_scorer = FinbertSentimentScorer()
finbert_scored = finbert_scorer.score_all(headlines)

for s in finbert_scored:
    print(f"[{s.sentiment_score:+d}] ({s.confidence:.2f}) {s.headline.title}")

In [ ]:
qwen_scorer = QwenSentimentScorer()
qwen_scored = qwen_scorer.score_all(headlines)

for s in qwen_scored:
    print(f"[{s.sentiment_score:+d}] {s.headline.title}")

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "title": f.headline.title,
            "finbert_label": f.label,
            "finbert_score": f.sentiment_score,
            "qwen_label": q.label,
            "qwen_score": q.sentiment_score,
            "agree": f.sentiment_score == q.sentiment_score,
        }
        for f, q in zip(finbert_scored, qwen_scored)
    ]
)
comparison_df

In [ ]:
agreement_rate = comparison_df["agree"].mean()
print(f"FinBERT and Qwen agree on {agreement_rate:.0%} of headlines for {TICKER}.")

## 4. Aggregate rating

`aggregate_rating()` averages a ticker's scored headlines into a single BUY/HOLD/SELL verdict — the same function the CLI and dashboard use.

In [ ]:
finbert_rating = aggregate_rating(TICKER, finbert_scored)
qwen_rating = aggregate_rating(TICKER, qwen_scored)

print(f"FinBERT: {finbert_rating.rating} (avg score {finbert_rating.avg_score:+.2f})")
print(f"Qwen:    {qwen_rating.rating} (avg score {qwen_rating.avg_score:+.2f})")

## 5. Visualizing the two models side by side

A quick bar chart makes disagreement easier to spot than scanning the table above.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(comparison_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width / 2, comparison_df["finbert_score"], width, label="FinBERT")
ax.bar(x + width / 2, comparison_df["qwen_score"], width, label="Qwen")
ax.set_xticks(x)
ax.set_xticklabels(
    [t[:30] + "..." if len(t) > 30 else t for t in comparison_df["title"]], rotation=75, ha="right"
)
ax.set_ylabel("Sentiment score (-1 / 0 / +1)")
ax.set_title(f"{TICKER}: FinBERT vs. Qwen, per headline")
ax.legend()
ax.axhline(0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.show()

## Ideas worth exploring next

Things this prototype surfaces that could become real features:

- **Model disagreement as its own signal.** When FinBERT and Qwen disagree on a headline, that's often a genuinely ambiguous or sarcastic one — worth surfacing in the dashboard rather than averaging it away.
- **Publisher weighting.** Not all sources are equally reliable; a Reuters headline and an obscure blog post currently count the same in `aggregate_rating()`.
- **Recency decay.** A two-week-old headline currently counts as much as one from this morning. A half-life weighting would make the rating more responsive to new information.
- **Backtesting ratings against forward returns.** `prices.py` already computes 1D/1W/QTD/YTD returns — pairing a day's rating with the *next* period's return would start to answer "does this sentiment score actually predict anything?"
- **Earnings-window sentiment drift.** Now that `earnings.py` knows the next earnings date, it'd be interesting to track whether sentiment systematically shifts in the days leading up to a report.
- **A third, larger model.** Qwen2.5-0.5B is tiny and fast but not very capable; swapping in a bigger instruct model (or a FinBERT-tone variant) would show whether scale changes the FinBERT/LLM agreement rate.
- **Rate limiting / caching yfinance calls.** Looping this notebook over a full watchlist quickly will hit Yahoo Finance's informal rate limits — worth building in before scaling this past one ticker at a time.